In [1]:
%cd ../.
import os, sys
sys.path.insert(0, os.path.expanduser('~/CDD_Vault_API/python'))  # CDD Vault API (get_df)

/home/gtamo/Px_interface


In [2]:
%load_ext autoreload
%autoreload 2

import os, re, gc, ctypes, json, time, importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from datetime import date
from rdkit import Chem
import yaml
import joblib
from tqdm import tqdm
from types import SimpleNamespace
tqdm.pandas()

# local modules
import python.functions as fn
from get_library import get_df   # CDD Vault collection export

In [3]:
## params — single source of truth in config/config.yaml.
## Loaded as a `config` namespace AND injected as globals, so both
## `config.RAW_PROTEOMICS_PATH` and bare `RAW_PROTEOMICS_PATH` work.

with open('config/config.yaml') as _f:
    _cfg = yaml.safe_load(_f)
config = SimpleNamespace(**_cfg)
globals().update(_cfg)
print(f'> loaded {len(_cfg)} params from config/config.yaml')

interface_output = DROPBOX_ML # GTLOCAL #  
print(DFRAW_PATH)

> loaded 37 params from config/config.yaml
/home/gtamo/Px_interface/output/MS/df_raw.parquet


## 0. Imports

### 0.1. Old lib

In [4]:
%%time
if DFRAW_OVERWRITE:
    ## Px part 1
    df_raw_20260429, MS20260429 = fn.load_proteomics_data(
        RAW_PROTEOMICS_PATH,
        CLEAN_PROTEOMICS_PATH,
        drop_plates=['Plate12', 'Plate15', 'Plate23'],
    )

    ## Px part 2:
    df_raw_20260520, MS20260520 = fn.load_proteomics_data(
        PX_20260520_DB,            # raw per-gene table (Database export) — was wrongly CDDVault
        PX_20260520_CDDVAULT,      # metadata table (Vault export: SMILES, Collections)
        drop_plates=['Plate12', 'Plate15', 'Plate23'],
        mode='cddvault',           # Collections recipe (drops PROTACs), join on 'Batch Molecule-Batch ID'
        collections=['AJ', 'AK'],
    )

    ## Px part 3:
    df_raw_20260529, MS20260529 = fn.load_proteomics_data(
        PX_20260529_DB,            # raw per-gene table (Database export) — was wrongly CDDVault
        PX_20260529_CDDVAULT,      # metadata table (Vault export: SMILES, Collections)
        drop_plates=['Plate12', 'Plate15', 'Plate23'],
        mode='cddvault',           # Collections recipe (drops PROTACs), join on 'Batch Molecule-Batch ID'
        collections=['AJ', 'AK'],
    )
    # manual formatting for 20260529 metadata:
    MS20260529 = MS20260529.rename(columns={'Molecule-Batch ID':'Batch Molecule-Batch ID','Nr. Down':'MSData - Proteomics activities: Nr. Down',"Cmpd Activity"	: "MSData - Proteomics activities: Cmpd Activity"})
    parts = MS20260529['Batch Molecule-Batch ID'].str.split('-', n=2, expand=True)
    MS20260529['Molecule Name'] = parts[0] + '-' + parts[1]   # 'SRB-0000385'
    MS20260529['batch']    = parts[2]                    # '001'

    ## Making final df_raw
    df_raw = pd.concat([df_raw_20260429,df_raw_20260520,df_raw_20260529]).reset_index(drop=True)
    # df_raw[['genes']].drop_duplicates().to_csv('data/MS/Px_genes.csv')

    ## check whether all compounds are including in the df_raw:

    colskeep = ['Molecule Name','MSData - Proteomics activities: Nr. Down','origin',"MSData - Proteomics activities: Cmpd Activity"]
    MS = pd.concat([MS20260429.assign(origin='MS20260429'),
                    MS20260520.assign(origin='MS20260520'),
                    MS20260529.assign(origin='MS20260529')]).reset_index(drop=True)[colskeep].rename(columns={'Molecule Name':'compound',
                                                                                                            'MSData - Proteomics activities: Nr. Down':'ndown',
                                                                                                            "MSData - Proteomics activities: Cmpd Activity":'activity'})
    MS['date'] = pd.to_datetime(MS['origin'].str.replace('MS', ''))
    
    # save a local copy of old lib df_raw and MS (temporary)
    MS.to_parquet(MS_PATH, index=False)
    df_raw.to_parquet(DFRAW_PATH, index=False)

    del df_raw_20260429,df_raw_20260520,df_raw_20260529
else:
    df_raw = pd.read_parquet(DFRAW_PATH)
    MS     = pd.read_parquet(MS_PATH)


CPU times: user 6.7 s, sys: 14.6 s, total: 21.3 s
Wall time: 3.6 s


In [5]:
## OpenTargets disease scores for every gene seen in the proteomics data.
## Cached to parquet — re-run is instant. Delete the file (or change OT_CACHE)
## to force a recompute from the local bulk dump under data/external/opentarget/.

if os.path.exists(OT_CACHE):
    print(f'> loading cached {OT_CACHE}')
    ot_df = pd.read_parquet(OT_CACHE)
else:
    # normalise the gene column: explode multi-gene strings (peptides mapping to >1
    # protein), strip whitespace, drop empties / NaNs, dedupe.
    unique_genes = pd.DataFrame({
        'gene': (df_raw['genes'].dropna().astype(str)
                  .str.split(r'[;,|]', regex=True).explode()
                  .str.strip()
                  .replace('', pd.NA).dropna()
                  .unique())
    })
    print(f'> {len(unique_genes):,} unique gene symbols in df_raw')

    ot_df = fn.get_opentarget_disease_score(
        unique_genes, gene_col='gene',
        top_n=20,
        ot_root=OT_ROOT,
    )
    os.makedirs(os.path.dirname(OT_CACHE), exist_ok=True)
    ot_df.to_parquet(OT_CACHE, index=False)
    print(f'> wrote {OT_CACHE}')

print(f'> {len(ot_df):,} (target, disease) rows  /  '
      f'{ot_df["target_symbol"].nunique():,} targets resolved')
ot_df.head(2)

## disease areas of interest to big pharma:
PRIORITY = {
    # Annotations: flagship products per pharma, just to anchor priority intuition.
    'cancer or benign tumor',                       # universal — BMS Opdivo/Yervoy, Roche Herceptin/Tecentriq/Phesgo, NVS Kisqali/Kymriah/Pluvicto, Pfizer Ibrance/Padcev, Lilly Verzenio/Jaypirca
    'hematologic disease',                          # BMS Revlimid/Pomalyst (multiple myeloma), Pfizer Elrexfio, NVS Tasigna, Roche Hemlibra (haemophilia), Lilly Jaypirca
    'cardiovascular disease',                       # BMS Eliquis, NVS Entresto/Leqvio (PCSK9 siRNA), Pfizer Vyndaqel (TTR amyloidosis)
    'immune system disease',                        # BMS Zeposia/Sotyktu/Orencia, NVS Cosentyx/Xolair, Pfizer Xeljanz/Cibinqo, Lilly Olumiant/Taltz, Roche Actemra
    'musculoskeletal or connective tissue disease', # RA / lupus / psoriasis (overlaps immune) — BMS Orencia, NVS Cosentyx, Pfizer Xeljanz, Lilly Olumiant
    'nervous system disease',                       # NVS Kesimpta/Gilenya (MS), Roche Ocrevus/Evrysdi (MS, SMA), Lilly Kisunla (AD, donanemab), BMS Cobenfy (post-Karuna)
    'psychiatric disorder',                         # BMS Cobenfy (schizophrenia), Lilly historic SSRIs/atypicals, otherwise smaller pharma footprint
    'nutritional or metabolic disease',             # Lilly #1 (Mounjaro / Zepbound — obesity/T2D), Pfizer & NVS chasing GLP-1 follow-ons
    'endocrine system disease',                     # Lilly (hormone/metabolic overlap), oncology adjacency (hormone-driven cancers) for BMS/Roche
    'disorder of visual system',                  # ← uncomment for Roche specifically (Vabysmo/Lucentis ophthalmology franchise)
    'respiratory or thoracic disease',            # ← uncomment for COPD/asthma plays (NVS Xolair, GSK; not named-four core)
    'infectious disease',                         # ← uncomment for vaccines/antivirals (Pfizer Comirnaty/Paxlovid/Prevnar)
}
DROP = {'phenotype', 'measurement', 'biological_process',
        'animal disease', 'medical procedure'}

def is_pharma_relevant(ta_str):
    """Keep rows whose therapeutic_areas overlap PRIORITY and aren't pure DROP-only."""
    if not isinstance(ta_str, str) or not ta_str:
        return False
    tokens = set(ta_str.split('|'))
    return bool(tokens & PRIORITY) and bool(tokens - DROP)

ot_pharma = ot_df[ot_df['therapeutic_areas'].apply(is_pharma_relevant)].reset_index(drop=True)
print(f'> {len(ot_pharma):,} / {len(ot_df):,} (target, disease) rows pass the BMS-style filter')
print(f'> covers {ot_pharma["target_symbol"].nunique():,} unique targets')

# breakdown by which priority area each row hits (a row can hit multiple)
print('\nrows per priority area:')
for token in sorted(PRIORITY):
    n = ot_pharma['therapeutic_areas'].str.split('|').apply(lambda s: token in s).sum()
    print(f'  {n:>6,}  {token}')

ot_pharma.head(2)

## get target list
ot_df_rank = (ot_pharma.sort_values('overall_score',ascending=False).reset_index(drop=True).groupby('target_symbol').first().sort_values('overall_score',ascending=False).reset_index())
target_list = list(ot_df_rank[ot_df_rank['overall_score'] > 0.4]['target_symbol'])

## add pharma target lists: BOTH the small-molecule patent file (gene col) AND
## the BMS pipeline file (hgnc_symbol col), so every pharma/BMS target is included.
patents = list(pd.read_csv(PHARMA_PATENT_CSV)['gene'].dropna().unique())
bms     = list(pd.read_csv(BMS_GENES)['hgnc_symbol'].dropna().unique())
target_list += patents + bms

target_list = list(dict.fromkeys(target_list))   # de-dupe, keep order
print(f'> target_list: {len(target_list)} unique '
      f'(OT>0.4 + {len(set(patents))} patent + {len(set(bms))} BMS genes)')
len(target_list)

> loading cached /home/gtamo/MS_ML/output/MS/opentargets_target_disease.parquet
> 232,004 (target, disease) rows  /  11,584 targets resolved
> 78,935 / 232,004 (target, disease) rows pass the BMS-style filter
> covers 9,105 unique targets

rows per priority area:
  17,071  cancer or benign tumor
   3,890  cardiovascular disease
   2,504  disorder of visual system
   3,466  endocrine system disease
   3,416  hematologic disease
   3,938  immune system disease
   3,710  infectious disease
   8,708  musculoskeletal or connective tissue disease
  25,393  nervous system disease
   9,849  nutritional or metabolic disease
   1,704  psychiatric disorder
   2,357  respiratory or thoracic disease
> target_list: 6706 unique (OT>0.4 + 172 patent + 46 BMS genes)


6706

In [6]:
## derive MS score
df_raw = df_raw.dropna()
df_raw['-log10(p-value)'] = -np.log10(df_raw['pvalue'])
df_raw['ms_score'] = (-df_raw['-log10(p-value)'] * df_raw['logfc']).clip(lower=0.0, upper=100.0)
df_raw = df_raw.sort_values('ms_score',ascending=False)

df_ms = df_raw[df_raw['significant']==1].groupby(['genes','MSPlate']).first().reset_index()

/home/gtamo/miniconda3/envs/ML/lib/python3.12/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: divide by zero encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)


#### 0.2. New lib

In [7]:
## load + concat every FBX tranche under FBX_DIR into the unified FBX tables. Tranche folders
## are auto-discovered (any date-named subdir holding *FBX_<KIND>*.csv) so a new tranche just
## needs its folder dropped in — no config/date edits. Disjoint uniquecontrast -> plain concat.
FBX_TRANCHES = sorted(t for t in (os.path.join(FBX_DIR, d) for d in os.listdir(FBX_DIR))
                      if os.path.isdir(t) and os.path.basename(t)[:8].isdigit()
                      and any('FBX_REPORT' in f for f in os.listdir(t)))
def _fbx_csv(tranche, kind):   # the one *FBX_<kind>*.csv in a tranche (tolerates a _02 re-export suffix)
    return os.path.join(tranche, next(f for f in os.listdir(tranche)
                                      if f'FBX_{kind}' in f and f.endswith('.csv')))
def _load_fbx(kind):
    return pd.concat([pd.read_csv(_fbx_csv(t, kind)) for t in FBX_TRANCHES], ignore_index=True)

FBX_MEASURE  = _load_fbx('MEASURE')
FBX_MSSCORE  = _load_fbx('MSSCORE')
FBX_REPORT   = _load_fbx('REPORT')
target2R2_df = pd.read_csv(GENE_SAR_OUT).rename(columns={'gene':'genes'})

# uniquecontrast -> compound (SRB-XXXXXXX, batch stripped); one map reused by every combine below
_p = FBX_REPORT['srbnumber'].astype(str).str.split('-', n=2, expand=True)
uc2compound = (FBX_REPORT.assign(compound=_p[0] + '-' + _p[1])
               .drop_duplicates('uniquecontrast').set_index('uniquecontrast')['compound'])

print(f'> FBX: {len(FBX_TRANCHES)} tranches | MEASURE {len(FBX_MEASURE):,} rows | '
      f'MSSCORE {len(FBX_MSSCORE):,} | REPORT {len(FBX_REPORT):,} '
      f'({FBX_REPORT["uniquecontrast"].nunique():,} experiments)')

> FBX: 6 tranches | MEASURE 14,209,538 rows | MSSCORE 15,695 | REPORT 1,696 (1,696 experiments)


In [8]:
# FBX: 5 batches | MEASURE 9,889,826 rows | MSSCORE 10,542 | REPORT 1,168 (1,168 experiments)

#### 0.3 combine all datasets:

In [9]:
## 1. combine df_raw & FBX_MEASURE
## Unified per-(compound x gene x experiment) MEASURE table for the whole Px set.
## df_raw (broad: 2,237 compounds x 86 plates x 12k genes) UNION FBX_MEASURE (the
## WT/KO 'Pw' condition plates). They overlap on 3 plates / 93 uniquecontrasts where
## logfc is the SAME measurement (corr 0.85, 94% identical) -> FBX kept as source of
## truth there: take ALL FBX rows + df_raw rows from experiments NOT in FBX.
_cols = ['compound', 'genes', 'pg', 'plate', 'uniquecontrast',
         'logfc', 'pvalue', 'adjpval', 'significant']

# FBX_MEASURE has no `compound` -> add it via the uc2compound map
fbx_std = (FBX_MEASURE.assign(compound=FBX_MEASURE['uniquecontrast'].map(uc2compound))
           .reindex(columns=_cols).assign(source='FBX'))

# df_raw -> same schema (MSPlate -> plate)
dr_std = df_raw.rename(columns={'MSPlate': 'plate'}).reindex(columns=_cols).assign(source='df_raw')

# FBX is source of truth on the shared experiments
_shared_uc = set(FBX_MEASURE['uniquecontrast']) & set(df_raw['uniquecontrast'].astype(str))
measure = pd.concat([fbx_std, dr_std[~dr_std['uniquecontrast'].astype(str).isin(_shared_uc)]],
                    ignore_index=True)

print(f'> combined MEASURE: {len(measure):,} rows')
print(f'  experiments (uniquecontrast): {measure["uniquecontrast"].nunique():,} '
      f'(FBX {fbx_std["uniquecontrast"].nunique()}, df_raw-only '
      f'{dr_std["uniquecontrast"].nunique() - len(_shared_uc)}, shared->FBX {len(_shared_uc)})')
print(f'  compounds {measure["compound"].nunique():,} | genes {measure["genes"].nunique():,} '
      f'| plates {measure["plate"].nunique():,}')
print(measure["source"].value_counts().to_string())
_no_cmp = int(fbx_std["compound"].isna().sum())
if _no_cmp:
    print(f'  note: {_no_cmp:,} FBX rows have no compound (uniquecontrast absent from FBX_REPORT '
          f'- control/QC contrasts); kept for volcanoes, excluded from the compound panel.')


> combined MEASURE: 37,758,399 rows
  experiments (uniquecontrast): 4,105 (FBX 1696, df_raw-only 2409, shared->FBX 93)
  compounds 3,537 | genes 12,041 | plates 122
source
df_raw    23548861
FBX       14209538
  note: 8,425 FBX rows have no compound (uniquecontrast absent from FBX_REPORT - control/QC contrasts); kept for volcanoes, excluded from the compound panel.


In [10]:
## 2. combine FBX_MSSCORE & df_ms:
## Unified per-(gene x plate) MS-SCORE table (feeds the dots). df_ms is the df_raw
## side (strongest-ms_score significant compound per gene-plate); FBX_MSSCORE is the
## curated side (+ OpenTargets association/genetic/literature scores). Same overlap
## rule as the measure combine: FBX is source of truth on shared (gene, plate).
## NB: re-read all FBX batches fresh from CSV so this cell is independent of the in-place
## edits above (the per-gene collapse drops 'plate').
_FBX_MS = _load_fbx('MSSCORE')
_FBX_MS = _FBX_MS[~_FBX_MS['plate'].isin(['Plate12', 'Plate15', 'Plate23'])]   # same noisy-plate drop
_cols = ['genes', 'plate', 'uniquecontrast', 'compound', 'pg', 'ms_score',
         'association_score', 'genetic_score', 'literature_score', 'activity',
         'logfc', 'pvalue', 'significant']

# FBX_MSSCORE -> one row per (gene, plate) [strongest ms_score], + compound via uc2compound
fbx_gp = (_FBX_MS.sort_values('ms_score', ascending=False)
          .groupby(['genes', 'plate'], as_index=False).first()
          .assign(compound=lambda d: d['uniquecontrast'].map(uc2compound))
          .reindex(columns=_cols).assign(source='FBX'))

# df_ms is already per (gene, plate) (MSPlate -> plate)
df_ms_std = df_ms.rename(columns={'MSPlate': 'plate'}).reindex(columns=_cols).assign(source='df_raw')

# FBX source of truth on shared (gene, plate)
_fbx_keys = set(map(tuple, fbx_gp[['genes', 'plate']].itertuples(index=False)))
_keep = ~df_ms_std.set_index(['genes', 'plate']).index.isin(_fbx_keys)
mscore = pd.concat([fbx_gp, df_ms_std[_keep]], ignore_index=True)

print(f'> combined MS-SCORE: {len(mscore):,} (gene,plate) rows '
      f'(FBX {len(fbx_gp):,}, df_raw {int(_keep.sum()):,})')
print(f'  genes {mscore["genes"].nunique():,} | plates {mscore["plate"].nunique():,}')
print(mscore["source"].value_counts().to_string())
print(f'  association/genetic/literature present for {int(mscore["association_score"].notna().sum()):,} rows '
      f'(FBX genes only; df_raw genes get these from OpenTargets in the next step).')
mscore.head(2)


> combined MS-SCORE: 49,040 (gene,plate) rows (FBX 12,207, df_raw 36,833)
  genes 8,782 | plates 114
source
df_raw    36833
FBX       12207
  association/genetic/literature present for 11,814 rows (FBX genes only; df_raw genes get these from OpenTargets in the next step).


,genes,plate,uniquecontrast,compound,pg,ms_score,association_score,genetic_score,literature_score,activity,logfc,pvalue,significant,source
0,A1BG,Pw127,SRB.0010320.001_vs_SRB.0010320.001_complement_...,SRB-0010320,P04217,15.109094,0.343864,0.565631,0.614010,High (>25),NaN,NaN,NaN,FBX
1,AADAC,Pw101,SRB.0007177.001_vs_SRB.0007177.001_complement_...,SRB-0007177,P22760,4.637844,0.352390,0.579656,0.545862,High (>25),NaN,NaN,NaN,FBX


In [11]:
## 3. combine MS & FBX_REPORT  ->  source-derived per-experiment REPORT
## MS is just the compound-level collapse of CLEAN + PX_20260520 + PX_20260529, so we
## derive the df_raw-side REPORT straight from those source tables at the per-EXPERIMENT
## grain -> real Concentration (uM) + per-plate activity / nr_down / cell_line / condition.
## Join to df_raw's uniquecontrast via (MoleculeBatchID, MSPlate) [99.7% match]; MS is
## only a fallback for activity on the rare unmatched experiments. Then union with
## FBX_REPORT, FBX source of truth on shared uniquecontrasts.
P = 'MSData - Proteomics activities: '
def _load_src(path, prefixed):
    pre = P if prefixed else ''
    m = {('Batch Molecule-Batch ID' if prefixed else 'Molecule-Batch ID'): 'batch',
         pre + 'MSPlate': 'plate',
         (P + 'Concentration (uM)' if prefixed else 'Concentration'): 'concentration',
         pre + 'Cmpd Activity': 'activity', pre + 'Nr. Down': 'nr_down',
         pre + 'Cell line': 'cell_line', pre + 'Sample Condition': 'condition'}
    cols = pd.read_csv(path, nrows=0).columns
    m = {k: v for k, v in m.items() if k in cols}
    return pd.read_csv(path, usecols=list(m)).rename(columns=m)

SRC = pd.concat([_load_src(CLEAN_PROTEOMICS_PATH, True),
                 _load_src(PX_20260520_CDDVAULT, True),
                 _load_src(PX_20260529_CDDVAULT, False)], ignore_index=True)
SRC = (SRC.dropna(subset=['batch', 'plate']).drop_duplicates(['batch', 'plate'])
       .rename(columns={'batch': 'MoleculeBatchID', 'plate': 'MSPlate'}))

_cols = ['uniquecontrast', 'compound', 'plate', 'concentration', 'activity',
         'nr_down', 'cell_line', 'condition']

# df_raw side: experiments joined to source metadata on (MoleculeBatchID, MSPlate)
rep_dr = (df_raw[['uniquecontrast', 'MoleculeBatchID', 'MSPlate', 'compound']].drop_duplicates('uniquecontrast')
          .merge(SRC, on=['MoleculeBatchID', 'MSPlate'], how='left')
          .rename(columns={'MSPlate': 'plate'}))
# MS fallback for the rare unmatched activities (compound-level, latest tranche)
_msact = MS.sort_values('date').drop_duplicates('compound', keep='last').set_index('compound')['activity']
rep_dr['activity'] = rep_dr['activity'].fillna(rep_dr['compound'].map(_msact))
rep_dr = rep_dr.reindex(columns=_cols).assign(source='df_raw')

# FBX side: compound via uc2compound
rep_fbx = (FBX_REPORT.assign(compound=FBX_REPORT['uniquecontrast'].map(uc2compound))
           .reindex(columns=_cols).drop_duplicates('uniquecontrast').assign(source='FBX'))

# FBX source of truth on shared uniquecontrasts
_shared_uc = set(FBX_REPORT['uniquecontrast']) & set(df_raw['uniquecontrast'].astype(str))
report = pd.concat([rep_fbx, rep_dr[~rep_dr['uniquecontrast'].astype(str).isin(_shared_uc)]],
                   ignore_index=True)

print(f'> combined REPORT: {len(report):,} uniquecontrasts | compounds {report["compound"].nunique():,} '
      f'| plates {report["plate"].nunique():,}')
print(report['source'].value_counts().to_string())
print(f'  concentration present: {100 * report["concentration"].notna().mean():.0f}% '
      f'| activity present: {100 * report["activity"].notna().mean():.0f}%')
print('  activity:', report['activity'].value_counts(dropna=False).to_dict())
report.head(2)


> combined REPORT: 4,105 uniquecontrasts | compounds 3,537 | plates 122
source
df_raw    2409
FBX       1696
  concentration present: 100% | activity present: 100%
  activity: {'Low (2-10)': 1933, 'Silent': 783, 'Single (1)': 662, 'Medium (11-25)': 386, 'High (>25)': 341}


,uniquecontrast,compound,plate,concentration,activity,nr_down,cell_line,condition,source
0,SRB.0005840.001_vs_SRB.0005840.001_complement_...,SRB-0005840,Pw81BCDEKO,10.0,Medium (11-25),11.0,HepG2,WildType,FBX
1,SRB.0005518.001_vs_SRB.0005518.001_complement_...,SRB-0005518,Pw105VMMLN,10.0,Low (2-10),8.0,HepG2,WildType,FBX


In [12]:
## per-plate experiment DATE (tranche-derived) -> enables date-based plate filtering.
## Each plate maps to one tranche; the 3 FBX/df_raw overlap plates (Pw50/63/64) resolve to
## FBX, matching the source-of-truth dedup in the combine cells. The real per-experiment Date
## col is inconsistent (>1/plate in the 0429 export) and absent for FBX, so the tranche label
## is the canonical plate date. Plates absent from the exports go in PLATE_DATE_OVERRIDES.
_DFRAW_DATE_SRC = [   # (date, source export, plate col)
    ('2026-04-29', CLEAN_PROTEOMICS_PATH, 'MSData - Proteomics activities: MSPlate'),
    ('2026-05-20', PX_20260520_DB, 'MSPlate'),   # DB export = same plate provenance as df_raw
    ('2026-05-29', PX_20260529_DB, 'MSPlate'),   # (Vault exports are AJ/AK-filtered -> miss plates)
]
plate2date = {}
for _d, _p, _c in _DFRAW_DATE_SRC:
    plate2date.update({pl: _d for pl in pd.read_csv(_p, usecols=[_c], dtype=str)[_c].dropna().unique()})
for _t in FBX_TRANCHES:   # FBX assigned last -> wins on the shared plates; date from the folder name
    _d = pd.to_datetime(os.path.basename(_t)[:8]).strftime('%Y-%m-%d')
    plate2date.update({pl: _d for pl in pd.read_csv(_fbx_csv(_t, 'REPORT'),
                                                    usecols=['plate'])['plate'].dropna().astype(str).unique()})
plate2date.update(PLATE_DATE_OVERRIDES)   # manual fixes for plates absent from the exports

for _df in (measure, mscore, report):
    _df['date'] = pd.to_datetime(_df['plate'].astype(str).map(plate2date))
_unmapped = sorted(set(report.loc[report['date'].isna(), 'plate'].astype(str)))
print(f"> plate dates: {len(plate2date)} plates mapped | report spans "
      f"{report['date'].min():%Y-%m-%d} .. {report['date'].max():%Y-%m-%d}")
if _unmapped:
    print(f"  {len(_unmapped)} plate(s) date-less (add to PLATE_DATE_OVERRIDES): {_unmapped}")

> plate dates: 136 plates mapped | report spans 2026-04-29 .. 2026-06-30


## 1. Build Interface

In [13]:
## latest library straight from CDD Vault (collections AJ/AK), compound name + smiles only
if CHEMLIB_OVERWRITE:
    serac_df = (get_df(vault=7108, collections=['AK', 'AJ'], columns=['name', 'smiles','Px_repetition(yes/no)','Px_validated_WT(yes/no)','Px_Ligase_dependent(yes/no)',
                                                                      'Px_NameLigase_dependent','Px_Target_info','Px_Target_interest' ])
                .rename(columns={'name': 'compound'}))
    serac_df.to_csv(CHEMLIB_PATH,sep=',',index=False)
else:
    serac_df = pd.read_csv(CHEMLIB_PATH)

serac_df = serac_df.drop_duplicates()
for _c in ['Px_validated_WT(yes/no)', 'Px_Ligase_dependent(yes/no)','Px_repetition(yes/no)']:   # 'yes'/'no'/'' -> 1/0/NaN
    serac_df[_c] = serac_df[_c].astype('string').str.strip().str.lower().map({'yes': 1, 'no': 0})

## extract list of validated and devalidated targets
l1 = list(set(serac_df[(serac_df['Px_Ligase_dependent(yes/no)']==0) & 
                       (serac_df['Px_Target_interest'].notnull())]['Px_Target_interest']))
devalidated_targets = list(set([s.split(' ')[0].upper() for x in l1 for s in x.split(';')]))

l2 = list(set(serac_df[(serac_df['Px_Ligase_dependent(yes/no)']==1) & 
                       (serac_df['Px_Target_interest'].notnull())]['Px_Target_interest']))
validated_targets = list(set([s.split(' ')[0].upper() for x in l2 for s in x.split(';')]))

## extract list of validated and devalidated compounds:
devalidated_compounds = list(set(serac_df[(serac_df['Px_Ligase_dependent(yes/no)']==0) & 
                       (serac_df['Px_Target_interest'].notnull())]['compound']))
validated_compounds = list(set(serac_df[(serac_df['Px_Ligase_dependent(yes/no)']==1) & 
                       (serac_df['Px_Target_interest'].notnull())]['compound']))

print(f'> target: {len(validated_targets)} validated - {len(devalidated_targets)} devalidated targets')
print(f'> compound: {len(validated_compounds)} validated - {len(devalidated_compounds)} devalidated targets')
serac_df.shape

> target: 19 validated - 1117 devalidated targets
> compound: 33 validated - 133 devalidated targets


(8935, 8)

In [14]:
serac_df[serac_df['compound']=='SRB-0006761']

,compound,smiles,Px_repetition(yes/no),Px_validated_WT(yes/no),Px_Ligase_dependent(yes/no),Px_NameLigase_dependent,Px_Target_info,Px_Target_interest


In [15]:
## local params:
control_compounds = CONTROLS

## contaminants compounds to remove:
contaminants = list(pd.read_csv(CONTAMINANTS)['Molecule Name'])

## degradation research per target (one record/gene); file path lives in config.GENE_RESEARCH
with open(config.GENE_RESEARCH) as _f:   # path from config/config.yaml
    GENE_RESEARCH = json.load(_f)
print(f'> loaded degradation research for {len(GENE_RESEARCH)} genes; '
      f'fields: {list(GENE_RESEARCH[0].keys())}')

> loaded degradation research for 10191 genes; fields: ['gene_name', 'target_class', 'lof_therapeutic_benefit', 'degrader_vs_inhibitor_rationale', 'degrader_feasibility', 'depmap_dependency', 'opentargets_top_indications', 'existing_degraders', 'safety_flags', 'confidence', 'biology_rationale', 'sources']


In [16]:
## save or load interface data
## IFACE_OVERWRITE=True  -> build the render inputs from the unified tables (measure / mscore /
##   report / plate2date) and SAVE the four to IFACE_DIR. IFACE_OVERWRITE=False -> LOAD them and
##   free the heavy upstream frames, so you can skip the combine + build cells (10-15, 21) and run
##   only: config + the small param cells (contaminants / targets / research) + this + the render.
_IFACE_KEYS = ['iface_df', 'compounds_df', 'meas', 'plate2date']

if IFACE_OVERWRITE:
    DROP_PLATES = ['Plate12', 'Plate15', 'Plate23']
    # --- volcano source = unified measure (drop noisy plates) + p-value floor ---
    # a 0.0 p plots at y=300 under -log10; floor zeros to the smallest non-zero p so the
    # renderers' 1e-300 clip is inert (same as the Re-Compute Volcanoes cell). No-op if floored.
    meas = measure[~measure['plate'].isin(DROP_PLATES)].copy()
    _pmin = measure.loc[measure['pvalue'] > 0, 'pvalue'].min()
    meas.loc[meas['pvalue'].eq(0.0), 'pvalue'] = _pmin

    # --- gene-level axes: x=R2 (SAR full-genome), y=OpenTargets association + top area, z=ms_score ---
    R2_df = pd.read_csv(GENE_SAR_OUT)[['gene', 'R2']].rename(columns={'gene': 'genes'})
    if 'ot_df' not in globals():
        ot_df = pd.read_parquet(OT_CACHE)
    assoc = ot_df.groupby('target_symbol')['overall_score'].max().rename('association_score')
    PRIORITY = ['cancer or benign tumor', 'hematologic disease', 'cardiovascular disease',
                'immune system disease', 'musculoskeletal or connective tissue disease',
                'nervous system disease', 'psychiatric disorder',
                'nutritional or metabolic disease', 'endocrine system disease']
    _rank = {a: i for i, a in enumerate(PRIORITY)}
    _areas = (ot_df[['target_symbol', 'overall_score', 'therapeutic_areas']]
              .assign(area=lambda d: d['therapeutic_areas'].fillna('').str.split('|')).explode('area'))
    _areas = _areas[_areas['area'].isin(_rank)].copy()
    _areas['_rank'] = _areas['area'].map(_rank)
    _top_area = (_areas.sort_values(['target_symbol', '_rank', 'overall_score'], ascending=[True, True, False])
                 .drop_duplicates('target_symbol', keep='first')
                 .rename(columns={'target_symbol': 'gene', 'area': 'disease_area'})[['gene', 'disease_area']])
    ms_gene = mscore.groupby('genes')['ms_score'].max().rename('ms_score')

    # --- dots: one per gene over the mscore universe; R2/association left-joined (missing -> 0.0) ---
    iface_df = (ms_gene.reset_index()
                .merge(R2_df, on='genes', how='left')   # left: keep genes lacking a SAR R2 (n_compounds < min_compounds / not yet computed)
                .merge(assoc, left_on='genes', right_index=True, how='left')
                .rename(columns={'genes': 'gene'})
                .merge(_top_area, on='gene', how='left'))
    iface_df[['R2', 'association_score']] = iface_df[['R2', 'association_score']].fillna(0.0)   # no SAR R2 / no OT association -> 0.0 (still plotted)
    _pharma = set(pd.read_csv(PHARMA_PATENT_CSV)['gene'].dropna().unique())
    _bms    = set(pd.read_csv(BMS_GENES)['hgnc_symbol'].dropna().unique())
    _model  = iface_df['R2'] > PHARMA_R2_CUTOFF
    iface_df.loc[iface_df['gene'].isin(_pharma) & _model, 'disease_area'] = 'pharma'
    iface_df.loc[iface_df['gene'].isin(_bms)    & _model, 'disease_area'] = 'BMS'
    print(f'> iface_df: {len(iface_df):,} gene dots (whole Px) | disease_area set for {iface_df["disease_area"].notna().sum():,}')

    # --- compounds_df: significant-down hits from measure + report metadata + smiles ---
    chemlib = pd.read_csv(CHEMLIB_PATH)[['compound', 'smiles']].drop_duplicates('compound')
    n_genes = meas.dropna(subset=['logfc', 'pvalue']).groupby('uniquecontrast')['genes'].nunique().rename('n_genes')
    rep = report[['uniquecontrast', 'compound', 'plate', 'concentration', 'activity']].drop_duplicates('uniquecontrast')
    _hit = meas.loc[(meas['significant'] == 1) & (meas['logfc'] < 0), ['genes', 'uniquecontrast', 'logfc', 'pvalue']]
    hits = _hit.merge(rep, on='uniquecontrast', how='left').rename(columns={'genes': 'gene'})
    hits = hits[hits['compound'].notna() & hits['compound'].str.startswith('SRB-')]
    hits = hits.sort_values(['gene', 'compound', 'plate', 'logfc'])
    compounds_df = (hits.groupby(['gene', 'compound', 'plate'], as_index=False).first()
                    .merge(chemlib, on='compound', how='left')
                    .merge(n_genes, on='uniquecontrast', how='left'))
    compounds_df = compounds_df[['gene', 'compound', 'plate', 'activity', 'n_genes',
                                 'uniquecontrast', 'logfc', 'smiles']]
    # keep only compounds present in serac_df (CDD AJ/AK library); exclude the rest from the viz
    _n0c = compounds_df['compound'].nunique()
    compounds_df = compounds_df[compounds_df['compound'].isin(chemlib['compound'])]
    print(f'> {compounds_df["compound"].nunique():,}/{_n0c:,} compounds present in serac_df (rest excluded)')
    # MoleculeBatchID (per experiment) for the volcano label text, sourced from df_raw.
    compounds_df['molecule_batch_id'] = compounds_df['uniquecontrast'].map(
        df_raw.drop_duplicates('uniquecontrast').set_index('uniquecontrast')['MoleculeBatchID'])
    # Experiments absent from df_raw (the …WT/KO/Eval plates) have no MoleculeBatchID there,
    # but the batch id is embedded in uniquecontrast (SRB.0005514.001_vs_… -> SRB-0005514-001);
    # reconstruct it for those rows, keeping it only when it matches the row's own compound.
    _miss = compounds_df['molecule_batch_id'].isna()
    _parsed = compounds_df.loc[_miss, 'uniquecontrast'].str.split('_vs_').str[0].str.replace('.', '-', regex=False)
    _valid = [p.startswith(c) for p, c in zip(_parsed, compounds_df.loc[_miss, 'compound'].astype(str))]
    compounds_df.loc[_miss, 'molecule_batch_id'] = _parsed.where(pd.Series(_valid, index=_parsed.index))
    _still = compounds_df['molecule_batch_id'].isna().sum()
    print(f'> molecule_batch_id reconstructed from uniquecontrast for {sum(_valid):,} rows; {_still:,} still missing')
    # drop Silent-activity experiments (no real down-modulation; shrinks the panel)
    _n0 = len(compounds_df)
    compounds_df = compounds_df[compounds_df['activity'] != 'Silent']
    print(f'> dropped {_n0 - len(compounds_df):,} Silent-activity rows -> {len(compounds_df):,} remain')
    print(f'> compounds_df: {len(compounds_df):,} (gene,compound,plate) rows across '
          f'{compounds_df["gene"].nunique():,} genes, {compounds_df["uniquecontrast"].nunique():,} volcanoes to render')

    # --- save the four render inputs ---
    os.makedirs(IFACE_DIR, exist_ok=True)
    iface_df.to_parquet(os.path.join(IFACE_DIR, 'iface_df.parquet'))
    compounds_df.to_parquet(os.path.join(IFACE_DIR, 'compounds_df.parquet'))
    meas.to_parquet(os.path.join(IFACE_DIR, 'meas.parquet'))
    with open(os.path.join(IFACE_DIR, 'plate2date.json'), 'w') as _fh:
        json.dump(plate2date, _fh)
    print(f'> saved interface inputs -> {IFACE_DIR}/ ({", ".join(_IFACE_KEYS)})')
else:
    iface_df     = pd.read_parquet(os.path.join(IFACE_DIR, 'iface_df.parquet'))
    compounds_df = pd.read_parquet(os.path.join(IFACE_DIR, 'compounds_df.parquet'))
    meas         = pd.read_parquet(os.path.join(IFACE_DIR, 'meas.parquet'))
    with open(os.path.join(IFACE_DIR, 'plate2date.json')) as _fh:
        plate2date = json.load(_fh)
    print(f'> loaded interface inputs from {IFACE_DIR}/ | iface_df {iface_df.shape}, '
          f'compounds_df {compounds_df.shape}, meas {meas.shape}, {len(plate2date)} plate dates')
    # free the heavy upstream frames (fully absorbed into the loaded inputs); render needs only the four
    for _v in ['measure', 'mscore', 'report', 'FBX_MEASURE', 'FBX_MSSCORE', 'FBX_REPORT',
               'df_raw', 'MS', '_FBX_MS', 'SRC']:
        globals().pop(_v, None)
    gc.collect()
    try: ctypes.CDLL('libc.so.6').malloc_trim(0)   # return freed arenas to the OS (Linux)
    except Exception: pass

> iface_df: 8,782 gene dots (whole Px) | disease_area set for 6,384
> 2,817/2,852 compounds present in serac_df (rest excluded)
> molecule_batch_id reconstructed from uniquecontrast for 18,632 rows; 0 still missing
> dropped 25 Silent-activity rows -> 57,307 remain
> compounds_df: 57,307 (gene,compound,plate) rows across 8,287 genes, 3,209 volcanoes to render
> saved interface inputs -> output/interface/ (iface_df, compounds_df, meas, plate2date)


In [ ]:
## step 4 (render): the per-gene 3D interface for the WHOLE Px set.
## Inputs (iface_df / compounds_df / meas / plate2date) come from the "save or load interface
## data" cell. Dots = one per gene: x=R2 (SAR), y=association (OpenTargets), z=ms_score; volcano
## source = meas; compounds from compounds_df + smiles. Just colours / must-show / research here.
DISEASE_AREA_COLORS = {
    'pharma': ACTIVE_C, 'BMS': '#BA09D9',
    'cancer or benign tumor': '#DD870E', 'hematologic disease': '#FF0000',
    'cardiovascular disease': "#FB008A", 'immune system disease': '#2A9D8F',
    'musculoskeletal or connective tissue disease': '#264653',
    'nervous system disease': '#963802', 'psychiatric disorder': '#000000',
    'nutritional or metabolic disease': '#17E804', 'endocrine system disease': "#6EE5F5",
}
MUST_INCLUDE = sorted(iface_df.loc[iface_df['disease_area'].isin(['pharma', 'BMS']), 'gene'])
# Plates filter starts with only the latest tranche's plates ticked (newest experiments first);
# untick more dates in the panel to widen. ISO dates -> lexical max = newest.
_latest_date = max(plate2date.values())
PLATE_DEFAULTS = sorted(p for p, d in plate2date.items() if d == _latest_date)
print(f'> Plates default-ticked: {len(PLATE_DEFAULTS)} plate(s) on latest date {_latest_date}')
# gene_research = {gene_name: record}; tolerate a dict, a list of records, or a bad/stale value
# (-> empty; re-run the GENE_RESEARCH load cell to populate it).
_R = GENE_RESEARCH
if isinstance(_R, dict):
    gene_research = _R
elif isinstance(_R, (list, tuple)):
    gene_research = {r['gene_name']: r for r in _R if isinstance(r, dict) and 'gene_name' in r}
else:
    gene_research = {}
print(f'> gene_research: {len(gene_research)} genes' + ('' if gene_research else '  (empty - re-run the GENE_RESEARCH load cell)'))

# compound-panel cache (the ~30s 'compound panels' build): load it when not overwriting so
# the render is near-instant; rebuild + save when IFACE_OVERWRITE (referenced thumbnail/volcano
# files must already exist on disk). Lives with the render since only plot_3d_interface builds it.
_panels_path = os.path.join(IFACE_DIR, 'panels.json')
_panels_in = None
if not IFACE_OVERWRITE and os.path.exists(_panels_path):
    with open(_panels_path) as _f:
        _panels_in = json.load(_f)

fig, highlighted, _panels = fn.plot_3d_interface(
    iface_df,
    x_col='R2', y_col='association_score', z_col='ms_score',
    x_label='SAR predictability', y_label='association score', z_label='MS score',
    must_include=MUST_INCLUDE, top_n_highlight=40,
    compounds_df=compounds_df, plate_dates=plate2date, plate_defaults=PLATE_DEFAULTS,  # nested-by-date Plates filter; default-tick latest date only
    panels=_panels_in, return_panels=True,  # skip/cache the compound-panel build
    volcano_source=meas, volcano_key='uniquecontrast', page_size=5,
    png_dir=SRB_PNG_DIR,   # real compound PNGs from config; RDKit-render fallback when absent
    thumb_external=True,   # reference srb_png/<compound>.png next to the HTML (not inline base64)
    range_sliders=True, range_defaults={'x': 0.0, 'y': 0.0, 'z': 0.0}, # {SAR, OT, MS} # , {'x': 0.0, 'y': 0.0, 'z': 30} # {'x': 0.1, 'y': 0.5, 'z': 10}
    activity_defaults=['Single', 'Low'],   # focus view: R2>0.1, assoc>0.6, MS>10 # 
    control_compounds=control_compounds, control_default_on=False,  # hide controls by default
    contaminant_compounds=contaminants, contaminant_default_on=False,  # hide contaminants by default
    gene_research=gene_research,
    validated_targets=validated_targets, devalidated_targets=devalidated_targets,  # Target validation (Y/N) tickboxes
    validated_label='FBXO31 dependent', devalidated_label='FBXO31 independent',
    validated_compounds=validated_compounds, devalidated_compounds=devalidated_compounds,  # Compound validation tickboxes
    compound_validated_label='FBXO31 dependent', compound_devalidated_label='FBXO31 independent',
    depmap_defaults=['Selective', 'Non-essential'], conf_defaults=['High', 'Med'],
    lof_defaults=['Yes'], validation_defaults=None,  # default ticked boxes on load (validation: all)
    volcano_significant=True, volcano_dir=os.path.join(interface_output, 'interfaces', 'volcanoes_px'),
    volcano_n_jobs=16, volcano_xlim=(-8, 8), volcano_size_px=350,
    disease_area_colors=DISEASE_AREA_COLORS, nb_display=False,
    html_path=os.path.join(interface_output, 'interfaces', 'Serac_Px_interface.html'), # 20260612_3d_interface_PX_R2_assoc_ms.html
)

if IFACE_OVERWRITE or not os.path.exists(_panels_path):
    with open(_panels_path, 'w') as _f:
        json.dump(_panels, _f)
    print(f'> saved compound panels -> {_panels_path}')

> Plates default-ticked: 11 plate(s) on latest date 2026-06-30
> gene_research: 10191 genes
> 8,782 / 8,782 genes after filtering (x=R2, y=association_score, z=ms_score)
  [highlight] corner-top-40=40, association_score>None: 0, must=25, union=63
  [range_sliders] panels built for all 8782 plotted genes


compound panels: 100%|██████████| 8211/8211 [01:59<00:00, 68.57gene/s] 


> long-format panel: 55,432 compound entries across 8211 genes with compounds (page_size=5, 122 plates)
> built 55,432 structure thumbnails across 8782 genes (png=55432, rdkit=0, missing=0)


volcano cache scan: 100%|██████████| 57209/57209 [00:00<00:00, 451785.64task/s]


> volcanoes: 57,209 cached, 0 rendered [interactive SVG] -> /mnt/c/Users/gtamo/Serac Biosciences Dropbox/Serac_team/4_Data_Sciences/15_ML/interfaces/volcanoes_px
  [range_sliders] default box ≈ 8782 genes; gene labels shown while ≤ 1000 in range (drag handles to widen/narrow)
  [activity_defaults] focus view starts on ['Low (2-10)', 'Single (1)']
  [gene_research] 7303/10191 records kept (plotted genes only) — 18.8 MB injected
wrote /mnt/c/Users/gtamo/Serac Biosciences Dropbox/Serac_team/4_Data_Sciences/15_ML/interfaces/Serac_Px_interface.html  (2.5 MB main doc)  +  Serac_Px_interface_data.js  (32.5 MB, deferred)
> saved compound panels -> output/interface/panels.json


In [ ]:
# compounds_df[compounds_df['compound']=='SRB-0010734']

In [ ]:
# select_genes = highlighted[ (highlighted['ms_score']>10)  & (highlighted['R2']>0.1) & (highlighted['association_score']>0.35)]
# select_genes[['gene']].to_csv('/mnt/c/Users/gtamo/Desktop/GT/disease_association/20260611_select_genes.csv',index=False,sep=',')

In [ ]:
# ## Re-(Compute) Volcanoes — one-off cache-refresh utility (logic wrapped in fn). Run ONLY when an
# ## EXISTING cached experiment's p-values changed; NEW experiments are rendered fresh (already
# ## floored) by step 4. Floors 0.0 p-values to the smallest non-zero p and overwrites the affected
# ## cached images. volcano_dir / xlim / size_px MUST match the step-4 plot_3d_interface call.
# _v = fn.floor_zero_pvalues_and_refresh_volcanoes(
#     measure, os.path.join(interface_output, 'interfaces', 'volcanoes_px'),  # MUST match step-4 volcano_dir (what the HTML loads)
#     drop_plates=['Plate12', 'Plate15', 'Plate23'],
#     xlim=(-8, 8), size_px=350, n_jobs=8)
# if _v['stats'] is None:
#     print('> no zero p-values in measure — nothing to floor/recompute '
#           '(already floored? re-run the build cell to restore them)')
# else:
#     print(f"> floored {_v['n_floored']:,} zero p-values across {_v['n_experiments']:,} experiments "
#           f"-> pmin={_v['pmin']:.3e} (-log10={-np.log10(_v['pmin']):.1f}); recomputed {_v['n_pairs']:,} "
#           f"volcanoes; measure floored in place — re-run step 4 to rebuild")

In [ ]:
iface_df[iface_df['gene']=='GNAS']

In [ ]:
MS[MS['compound']=='SRB-0010226']

In [ ]:
tmp = pd.read_csv('data/advanteidge/20260617/20260617_FBX_MSSCORE.csv')
tmp[tmp['genes']=='TGFB1']